# PSC walkthrough: from the bulk file to per-company features

This notebook walks a new user from the Companies House **PSC** (persons with significant control)
bulk file to usable tables and per-company features. It calls the project's own pipeline code at
every step rather than reimplementing parsing or feature definitions.

## 1. What the PSC data is

The **PSC register** records who ultimately owns or controls each UK company — the people (or other
companies) with significant control, and, where no such person is identified, the *statement* the
company filed instead. Companies House publishes the whole register once a day as a **bulk
snapshot**: 32 newline-delimited JSON (NDJSON) files, together about 16 million records.

The source is personal data and a snapshot, not an event log. This notebook uses the project's
**governed per-record load**, which removes direct names, exact birth dates and full addresses but
still retains linkable identifiers and transformed attributes. It is sensitive internal data, not
an anonymous or public table. Corrections over time are lost unless snapshots are archived.

For the dataset page and deeper guide, see
[`../docs/site/datasets/psc.qmd`](../docs/site/datasets/psc.qmd) and
[`../docs/psc-data-guide.md`](../docs/psc-data-guide.md). The per-company feature table is documented
in [`../docs/psc-bulk-features.md`](../docs/psc-bulk-features.md).


## 2. Setup

Use the repository's Python environment and install the checkout in editable mode before starting
the kernel (for this repository, `conda run -n env1 python -m pip install -e '.[dev]'`). The kernel
may start in either the repository root or `notebooks/`; the next cell resolves the root explicitly
and derives every repository data, documentation and script path from it.

Required environment and existing inputs:

- `PSC_PERSON_KEY_SECRET`: required; set it in the shell or the repository's gitignored `.env`.
- `PSC_SNAPSHOT_DIR` (optional override): directory containing all 32 verified snapshot ZIP files
  and their extracted TXT members. Default: `~/Downloads/psc/2026-09-25`.
- `PSC_MANIFEST` (optional override): manifest created by `ukcompany-psc fetch`. Default:
  `data/psc/2026-09-25/manifest.json` under the repository root.
- `PSC_REGISTER_CSV` (full coverage rerun only): the existing 2026-09-01 register CSV. No input is
  downloaded by this notebook.

One switch, `SAMPLE`, controls processing:

- `SAMPLE = True` (default) consumes part 1 only, after checking the complete 32-part ZIP archive
  and checking that the consumed extracted TXT matches its verified ZIP member.
- `SAMPLE = False` requires and consumes all 32 TXT parts. It cannot build features unless the
  loader found the totals record and every totals reconciliation passed.

Sample and full artifacts are written under separate `data/psc/walkthrough/.../sample` and
`.../full` directories. Nothing is written to publication or staging paths.


In [ ]:
import os
import sys
import time
from pathlib import Path

from ukcompany.cli import load_dotenv
from ukcompany.psc.walkthrough import (
    display_count as disp,
    require_feature_build_ready,
    resolve_repository_root,
    run_checked,
    validate_snapshot_inputs,
)

REPO_ROOT = resolve_repository_root(Path.cwd())
load_dotenv(REPO_ROOT / ".env")


def configured_path(variable, default):
    path = Path(os.environ.get(variable, default)).expanduser()
    return path if path.is_absolute() else REPO_ROOT / path


# The one processing switch.
SAMPLE = True
DATA_GOVERNANCE = True
assert DATA_GOVERNANCE, "this notebook is governed-only by design"

SNAPSHOT_DATE = "2026-09-25"
EXPECTED_PARTS = 32
PARTS_DIR = configured_path("PSC_SNAPSHOT_DIR", f"~/Downloads/psc/{SNAPSHOT_DATE}")
MANIFEST = configured_path(
    "PSC_MANIFEST", f"data/psc/{SNAPSHOT_DATE}/manifest.json"
)
REGISTER_CSV = configured_path(
    "PSC_REGISTER_CSV", "~/Downloads/BasicCompanyDataAsOneFile-2026-09-01.csv"
)
DOCS_DIR = REPO_ROOT / "docs"
SCRIPTS_DIR = REPO_ROOT / "scripts"

run_mode = "sample" if SAMPLE else "full"
RUN_DIR = REPO_ROOT / "data/psc/walkthrough" / SNAPSHOT_DATE / run_mode
LOAD_DIR = RUN_DIR / "governed-records"
FEAT_DIR = RUN_DIR / "governed-features"
SPILL_DIR = RUN_DIR / ".duckdb-spill"
COVERAGE_OUT = RUN_DIR / "psc-coverage-2026-09-25-vs-register-2026-09-01.md"

secret = os.environ.get("PSC_PERSON_KEY_SECRET")
print("repository root  :", REPO_ROOT)
print("run mode         :", run_mode)
print("data_governance  :", DATA_GOVERNANCE)
print("snapshot date    :", SNAPSHOT_DATE)
print("PSC_PERSON_KEY_SECRET is set:", bool(secret))  # never print the value


**Why a secret in governed mode?** The governed **per-record** schema is
transformed, not anonymous. It drops names, exact birth year/month, nationality, full addresses,
full postcodes and raw JSON, but retains company and PSC identifiers, country of residence,
five-year birth bands, postcode districts, and both HMAC-derived `person_key` and
`person_key_strict`. The keys support controlled linkage while avoiding storage of the source name
and birth fields; they do not eliminate linkage or re-identification risk. The loader therefore
requires `PSC_PERSON_KEY_SECRET` and the governed records must remain access-controlled.

The governed **per-company feature table** built later is a different, more aggregated product. It
does not contain either person key, birth bands, postcode districts or country of residence, and it
explicitly excludes both companies-per-person band columns.


## 3. Validate the existing snapshot

The repeatable fetch path is `ukcompany-psc refresh`, which downloads, verifies, extracts, loads and
reconciles a snapshot. This notebook does **not** download. It validates the existing manifest and
archive before loading:

1. the manifest date and exact `1..32 of 32` ZIP identities must be complete;
2. every ZIP must match its manifest size and SHA-256; and
3. every extracted TXT file the notebook will consume must hash to the corresponding member of the
   verified ZIP.

The third check closes an important gap: ZIP verification alone says nothing about a pre-existing
TXT file, and the idempotent extractor may reuse a same-sized TXT. Missing, partial or modified
inputs raise an actionable error before `load_psc` runs.


In [ ]:
if SAMPLE:
    parts_glob = str(PARTS_DIR / f"psc-snapshot-{SNAPSHOT_DATE}_1of{EXPECTED_PARTS}.txt")
else:
    parts_glob = str(PARTS_DIR / f"psc-snapshot-{SNAPSHOT_DATE}_*of{EXPECTED_PARTS}.txt")

input_check = validate_snapshot_inputs(
    MANIFEST,
    PARTS_DIR,
    parts_glob,
    SNAPSHOT_DATE,
    sample=SAMPLE,
    expected_parts=EXPECTED_PARTS,
)
print("validated complete ZIP archive parts:", input_check["manifest_parts"])
print("validated consumed TXT parts       :", input_check["selected_parts"])
print("ZIP SHA-256 and consumed TXT bytes : verified")


## 4. Load into governed per-record tables

`load_psc` parses the NDJSON into two Parquet tables and writes a load report:

- **`psc_records.parquet`** — one row per PSC, statement, exemption or totals record. Its governed
  schema still contains company/PSC identifiers, transformed quasi-identifiers and both person
  keys; it must remain access-controlled.
- **`psc_noc.parquet`** — one row per nature-of-control assertion, keyed by `record_id`.

The load report includes bad-line and written-file consistency checks. A complete snapshot also
contains one authoritative totals record. The loader raises on mismatches; this notebook adds a
second full-mode gate that refuses feature construction if totals reconciliation was skipped.


> **Cost warning.** Input validation hashes all 32 ZIP archives; full mode also
> reads every extracted TXT to compare it with its ZIP member. This is I/O-heavy (typically several
> minutes) but uses modest memory. The subsequent full load was last measured at about **11 minutes
> and ~10 GB RAM** (8 GB DuckDB limit plus overhead). Sample mode loads one validated TXT part.


In [ ]:
from ukcompany.psc.loader import load_psc

t0 = time.time()
report = load_psc(
    parts_glob,
    LOAD_DIR,
    SNAPSHOT_DATE,
    data_governance=DATA_GOVERNANCE,
    person_key_secret=secret,
    memory_limit_gb=8,
    spill_dir=str(SPILL_DIR),
)
print(f"loaded in {time.time() - t0:.0f}s   (mode={run_mode})")
print("lines:", disp(report["n_lines"]), " bad lines:", disp(report["n_bad_lines"]))
print("categories sum to lines:", report["categories_sum_to_lines"])


In [ ]:
import duckdb

for name in ("psc_records", "psc_noc"):
    path = LOAD_DIR / f"{name}.parquet"
    n = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
    cols = duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{path}')").fetchall()
    print(f"\n{name}.parquet — {disp(n)} rows, {len(cols)} columns")
    print("  columns:", ", ".join(column[0] for column in cols))


In [ ]:
# Every potentially identifying aggregate count uses the shared suppression helper.
print("records by kind (category_counts):")
for category, count in sorted(report["category_counts"].items(), key=lambda item: -item[1]):
    print(f"  {category:<14} {disp(count)}")

reconciliation = report.get("totals_reconciliation")
if reconciliation:
    print("\ntotals reconciliation:")
    for field, values in reconciliation.items():
        print(
            f"  {field}: parsed={disp(values['parsed'])} "
            f"CH={disp(values['ch_stated'])} match={values['match']}"
        )
else:
    print("\ntotals reconciliation: not present (allowed only in sample mode)")

print("bad-line reasons:")
for reason, count in report["bad_line_reasons"].items():
    print(f"  {reason:<16} {disp(count)}")


## 5. What a record looks like

To show the shape of a record without displaying anyone's real data, we use a **synthetic** record
(invented person, same structure as the project's test fixtures). The interesting part is the
`natures_of_control` list. Each entry is a long code like
`ownership-of-shares-75-to-100-percent-as-trust`; the shared `psc_natures` decomposition splits it
into three pieces:

- the **core right** — what kind of control it is (here, ownership of shares);
- the **band** — how much (here, 75–100%);
- the **suffix family** — on whose behalf (here, "as trust").

In [ ]:
from ukcompany.psc_natures import decompose_nature, classify_kind, summarise_natures

# Synthetic record — invented data, never from the real file.
synthetic = {
    "kind": "individual-person-with-significant-control",
    "name_elements": {"forename": "Alex", "surname": "Example"},  # shown only to prove it's fake
    "natures_of_control": [
        "ownership-of-shares-75-to-100-percent-as-trust",
        "voting-rights-75-to-100-percent",
        "right-to-appoint-and-remove-directors",
    ],
}

print("coarse kind:", classify_kind(synthetic["kind"]))
print()
for raw in synthetic["natures_of_control"]:
    p = decompose_nature(raw)
    print(f"{raw}\n    core={p.core!r}  band={p.band!r}  suffix={p.suffix_family!r}\n")

# summarise_natures rolls a set of codes into the per-company signals used later:
print("summary:", summarise_natures(synthetic["natures_of_control"]))

## 6. Build governed per-company features

`build_psc_features` aggregates the governed per-record tables to **one row per company**, reusing
the shared `psc_natures` definitions. This output is distinct from the governed records: it does
not retain person or PSC identifiers, country of residence, birth bands, postcode districts or
person keys. Governed feature mode also removes the two companies-per-person bands and asserts that
they are absent from the written schema. Every feature column is registered in `FIELD_DOCS`.


In [ ]:
from ukcompany.psc.features import build_psc_features, FEATURE_COLUMNS, GOVERNED_FEATURE_DROPPED
from ukcompany.derive import FIELD_DOCS

# Full mode must have found and passed the authoritative totals reconciliation.
require_feature_build_ready(report, full_snapshot=not SAMPLE)

t0 = time.time()
freport = build_psc_features(
    LOAD_DIR / "psc_records.parquet",
    LOAD_DIR / "psc_noc.parquet",
    FEAT_DIR,
    SNAPSHOT_DATE,
    data_governance=DATA_GOVERNANCE,
    memory_limit_gb=8,
    spill_dir=str(SPILL_DIR),
)
print(
    f"built features in {time.time() - t0:.0f}s   "
    f"companies: {disp(freport['n_companies'])}   (mode={run_mode})"
)
print("governed tier drops (person-linkage):", GOVERNED_FEATURE_DROPPED)


In [ ]:
# Each governed feature column with its FIELD_DOCS description.
docs_by_field = {d["field"]: d for d in FIELD_DOCS}
for col in FEATURE_COLUMNS:
    d = docs_by_field.get(col)
    if col == "company_number":
        print(f"{col:<32} the Companies House number (join key)")
    elif d:
        print(f"{col:<32} {d['definition']}")
    else:
        print(f"{col:<32} (not in FIELD_DOCS)")

## 7. Broad results

Everything displayed here is aggregate-only. Every positive count below 10 is rendered `<10>` by
the shared `display_count` helper imported during setup. In sample mode these are from one of 32
parts and describe shape, not whole-snapshot totals.


In [ ]:
FEAT = str(FEAT_DIR / "psc_company_features.parquet")
label = "PARTIAL SAMPLE (1/32 parts)" if SAMPLE else "FULL SNAPSHOT (32/32 parts)"
print("results basis:", label)


**Records by kind** — the mix of individuals, corporates, statements and so on.

In [ ]:
for k, v in sorted(report["category_counts"].items(), key=lambda kv: -kv[1]):
    print(f"  {k:<14} {disp(v)}")

**Active vs ceased** — a ceased PSC has stepped down but stays in the file for history.

In [ ]:
row = duckdb.sql(
    f"SELECT SUM(psc_n_records) r, SUM(psc_n_ceased) c FROM read_parquet('{FEAT}')"
).fetchone()
total, ceased = row[0] or 0, row[1] or 0
print(f"  active  {disp(total - ceased)}")
print(f"  ceased  {disp(ceased)}")

**Information state per company** — `identified` (at least one live PSC), `statement_only` (no PSC
but a filed statement) or `none_reported`.

In [ ]:
rows = duckdb.sql(
    f"SELECT psc_information_state, COUNT(*) n FROM read_parquet('{FEAT}') "
    "GROUP BY 1 ORDER BY n DESC"
).fetchall()
for state, n in rows:
    print(f"  {state:<16} {disp(n)}")

**Nature-of-control cores** — what kind of control is asserted, decomposed with `psc_natures`
(reusing `decompose_nature`, not re-counting raw strings).

In [ ]:
from collections import Counter
cores = Counter()
rows = duckdb.sql(f"SELECT DISTINCT right_raw FROM read_parquet('{LOAD_DIR / 'psc_noc.parquet'}')").fetchall()
# map each distinct code -> core via the shared function, then weight by occurrences
counts = dict(duckdb.sql(
    f"SELECT right_raw, COUNT(*) FROM read_parquet('{LOAD_DIR / 'psc_noc.parquet'}') GROUP BY 1"
).fetchall())
for code_str, n in counts.items():
    core = decompose_nature(code_str).core or "(unmapped)"
    cores[core] += n
for core, n in cores.most_common():
    print(f"  {core:<34} {disp(n)}")

**Maximum ownership band per company** — the largest share-ownership band each company reports.

In [ ]:
rows = duckdb.sql(
    f"SELECT COALESCE(psc_max_ownership_band, '(none)') b, COUNT(*) n "
    f"FROM read_parquet('{FEAT}') GROUP BY 1 ORDER BY n DESC"
).fetchall()
for band, n in rows:
    print(f"  {band:<22} {disp(n)}")

**ECCTA identity-verification counts** — how many PSC records are inside the new identity-checking
regime. **Point-in-time:** the rollout is ongoing, and `n_psc_id_statement_filed` / `_due` reflect
the *current* verification cycle (they can fall when it resets) — they are not lifetime totals.

In [ ]:
row = duckdb.sql(
    f"SELECT SUM(n_psc_id_verified), SUM(n_psc_id_verification_due), "
    f"SUM(n_psc_id_statement_filed) FROM read_parquet('{FEAT}')"
).fetchone()
for label_, v in zip(("verified (block present)", "verification due", "statement filed"), row):
    print(f"  {label_:<26} {disp(v or 0)}")

**Top statement codes** — what companies filed when no PSC is identified (suppressed `<10`).

In [ ]:
codes = Counter()
for (s,) in duckdb.sql(
    f"SELECT active_psc_statement_codes FROM read_parquet('{FEAT}') WHERE active_psc_statement_codes IS NOT NULL"
).fetchall():
    for c in s.split(","):
        codes[c] += 1
for code_str, n in codes.most_common(10):
    print(f"  {code_str:<48} {disp(n)}")

## 8. Historical coverage comparison

This section is explicitly the historical comparison **PSC 2026-09-25 versus register
2026-09-01**. It is not an aligned September join and is not the paused Handoff 10 join. Coverage
requires a full PSC feature table, so sample mode shows only the top-level, committed historical
summary from [`../docs/psc-coverage-2026-09-25.md`](../docs/psc-coverage-2026-09-25.md). It does not
display the category table, avoiding exposure of unsuppressed small cells in that older report.

Some company types are outside the PSC regime (for example CIOs, Registered Societies and Royal
Charter companies), so absence can be expected rather than a data gap. PSC records also include
dissolved companies that are absent from the register snapshot.


In [ ]:
historical_coverage = DOCS_DIR / "psc-coverage-2026-09-25.md"
if SAMPLE:
    # Top-level historical totals only; do not display the old unsuppressed category breakdown.
    lines = historical_coverage.read_text(encoding="utf-8").splitlines()
    print("\n".join(lines[:10]))
else:
    if not REGISTER_CSV.is_file():
        raise FileNotFoundError(
            f"register input missing: {REGISTER_CSV}; set PSC_REGISTER_CSV to the existing "
            "2026-09-01 Basic Company Data CSV"
        )
    command = [
        sys.executable,
        str(SCRIPTS_DIR / "psc_coverage_check.py"),
        "--features", str(FEAT_DIR / "psc_company_features.parquet"),
        "--register", str(REGISTER_CSV),
        "--register-date", "2026-09-01",
        "--snapshot-date", SNAPSHOT_DATE,
        "--out", str(COVERAGE_OUT),
    ]
    completed = run_checked(command, cwd=REPO_ROOT)
    print(completed.stdout.strip())
    print("historical comparison written to:", COVERAGE_OUT)


## 9. Governance and interpretation limits

- **Governed records remain linkable.** Removing names, exact birth dates and full addresses does
  not make the governed record tables anonymous. Company numbers, PSC/record identifiers, country
  of residence, five-year birth bands, postcode districts and both HMAC person keys remain. In
  combination with public source data they can support linkage or re-identification, so access and
  retention controls still apply.
- **Governed records differ from governed features.** The per-company feature table is aggregated
  and carries none of those per-record identifiers or transformed demographic/location fields.
- **No companies-per-person bands in governed features.** Both active and ever bands remain
  private/ungoverned features, because even a band can point to a person at a single-PSC company.
- **No nationality feature.** Nationality is removed from governed records and is not aggregated.
  Country of residence remains in the governed record schema but not in the per-company features.
- **`notified_on` is as filed.** It is not independently verified; treat it as a claim.

See the governance sections of [`../docs/psc-data-guide.md`](../docs/psc-data-guide.md) and
[`../docs/psc-bulk-features.md`](../docs/psc-bulk-features.md).

*Run basis: `SAMPLE = True` is a partial 1/32 sample. Set `SAMPLE = False` only with the complete,
verified inputs and sufficient time and memory described above.*
